In [1]:
# ============================================================
# Q4 — Missingness Indicator + Logistic Regression
# ============================================================

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. Load classification dataset
# ------------------------------------------------------------

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = data.target


print("Original shape:", X.shape)



Original shape: (569, 30)


In [2]:
# ------------------------------------------------------------
# 2. Artificially introduce missing values
# ------------------------------------------------------------

rng = np.random.RandomState(42)

X_missing = X.copy()

missing_rate = 0.10

mask = rng.rand(*X_missing.shape) < missing_rate

X_missing[mask] = np.nan


# ------------------------------------------------------------
# 3. Check missingness
# ------------------------------------------------------------

print("\nMissing percentage:")

print(
    X_missing.isnull().mean()
    .mul(100)
    .sort_values(ascending=False)
)


Missing percentage:
worst fractal dimension    14.586995
mean concave points        14.235501
mean texture               12.478032
mean concavity             12.126538
concave points error       11.423550
texture error              11.247803
smoothness error           11.247803
worst perimeter            11.072056
worst radius               10.896309
area error                 10.720562
worst compactness          10.544815
mean area                  10.544815
mean perimeter             10.369069
fractal dimension error    10.369069
mean radius                 9.841828
worst symmetry              9.841828
compactness error           9.666081
concavity error             9.666081
mean compactness            9.666081
mean fractal dimension      9.666081
mean symmetry               9.666081
perimeter error             9.666081
worst smoothness            9.490334
symmetry error              9.314587
worst texture               9.314587
radius error                9.138840
worst concavity  

In [3]:
# ------------------------------------------------------------
# 4. Train/test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_missing,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


# ============================================================
# MODEL A — MEDIAN ONLY
# ============================================================

median_pipeline = Pipeline([
    
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),
    
    (
        "model",
        LogisticRegression(
            max_iter=5000
        )
    )
])

In [4]:
# ------------------------------------------------------------
# 5. Train model
# ------------------------------------------------------------

median_pipeline.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 6. Predict
# ------------------------------------------------------------

median_probabilities = (
    median_pipeline
    .predict_proba(X_test)[:, 1]
)


# ------------------------------------------------------------
# 7. AUC
# ------------------------------------------------------------

median_auc = roc_auc_score(
    y_test,
    median_probabilities
)

print("\nMedian-only AUC:")
print(median_auc)



Median-only AUC:
0.9877645502645502


In [5]:
# ------------------------------------------------------------
# 8. Extract coefficients
# ------------------------------------------------------------

median_coefficients = (
    median_pipeline
    .named_steps["model"]
    .coef_[0]
)


coef_without_indicator = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": median_coefficients
})

print("\nCoefficients WITHOUT missingness indicators:")
display(coef_without_indicator)


Coefficients WITHOUT missingness indicators:


,Feature,Coefficient
0,mean radius,0.317671
1,mean texture,-0.031726
2,mean perimeter,0.012240
3,mean area,0.001240
4,mean smoothness,-0.224947
5,mean compactness,-0.437646
6,mean concavity,-0.706107
7,mean concave points,-0.331815
8,mean symmetry,-0.272306
9,mean fractal dimension,-0.050044


In [6]:

# ============================================================
# MODEL B — MEDIAN + MISSINGNESS INDICATOR
# ============================================================

indicator_pipeline = Pipeline([
    
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    
    (
        "model",
        LogisticRegression(
            max_iter=5000
        )
    )
])


# ------------------------------------------------------------
# 9. Train model
# ------------------------------------------------------------

indicator_pipeline.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 10. Predict
# ------------------------------------------------------------

indicator_probabilities = (
    indicator_pipeline
    .predict_proba(X_test)[:, 1]
)



In [7]:
# ------------------------------------------------------------
# 11. AUC
# ------------------------------------------------------------

indicator_auc = roc_auc_score(
    y_test,
    indicator_probabilities
)

print("\nMedian + Indicator AUC:")
print(indicator_auc)


# ============================================================
# 12. Extract coefficients
# ============================================================

indicator_coefficients = (
    indicator_pipeline
    .named_steps["model"]
    .coef_[0]
)


# ------------------------------------------------------------
# 13. Get names of generated features
# ------------------------------------------------------------

imputer = (
    indicator_pipeline
    .named_steps["imputer"]
)

indicator_features = (
    imputer
    .get_feature_names_out(X.columns)
)


coef_with_indicator = pd.DataFrame({
    "Feature": indicator_features,
    "Coefficient": indicator_coefficients
})


print("\nCoefficients WITH missingness indicators:")

display(coef_with_indicator)


Median + Indicator AUC:
0.9811507936507937

Coefficients WITH missingness indicators:


,Feature,Coefficient
0,mean radius,0.367857
1,mean texture,-0.049077
2,mean perimeter,0.010173
3,mean area,0.002649
4,mean smoothness,-0.203068
5,mean compactness,-0.413443
6,mean concavity,-0.632990
7,mean concave points,-0.294320
8,mean symmetry,-0.254213
9,mean fractal dimension,-0.054110


In [8]:
# ============================================================
# 14. Display only indicator coefficients
# ============================================================

indicator_rows = (
    coef_with_indicator[
        coef_with_indicator["Feature"]
        .str.contains("missingindicator")
    ]
)

print("\nMissingness Indicator Coefficients:")

display(indicator_rows)




Missingness Indicator Coefficients:


,Feature,Coefficient
30,missingindicator_mean radius,-0.074736
31,missingindicator_mean texture,0.160031
32,missingindicator_mean perimeter,-0.228330
33,missingindicator_mean area,0.532425
34,missingindicator_mean smoothness,-0.441481
35,missingindicator_mean compactness,-0.443088
36,missingindicator_mean concavity,0.479749
37,missingindicator_mean concave points,-0.578981
38,missingindicator_mean symmetry,-0.620500
39,missingindicator_mean fractal dimension,-0.755881


In [9]:
# ============================================================
# 15. Compare AUC
# ============================================================

auc_comparison = pd.DataFrame({
    
    "Model": [
        "Median Only",
        "Median + Missing Indicator"
    ],
    
    "AUC": [
        median_auc,
        indicator_auc
    ]
})

print("\nAUC Comparison:")

display(auc_comparison)


AUC Comparison:


,Model,AUC
0,Median Only,0.987765
1,Median + Missing Indicator,0.981151
